# Section 2 — Numerical Error and Floating-Point Arithmetic

[← Taylor Series](01_taylor_series.ipynb) · [Module 1](README.md) · [Next: Bisection →](03_bisection_method.ipynb)

Computers do not store arbitrary real numbers exactly. Numerical analysis therefore
tracks both the mathematical approximation error and the additional error introduced by
finite-precision arithmetic.

### Learning goals
- compute absolute and relative error;
- distinguish rounding from chopping;
- inspect the IEEE-754 binary64 representation used by Python `float`;
- recognize loss of significance from subtracting nearby numbers;
- reorganize a formula to reduce numerical error.

In [ ]:
import math
import struct

## Absolute and relative error

If \(p\) is the exact value and \(p^\ast\) is an approximation, then

\[
\text{absolute error}=|p-p^\ast|,
\qquad
\text{relative error}=\frac{|p-p^\ast|}{|p|}.
\]

In [ ]:
exact_value = math.pi
approximation = 3.14

absolute_error = abs(exact_value - approximation)
relative_error = absolute_error / abs(exact_value)

print(f"absolute error = {absolute_error:.10f}")
print(f"relative error = {relative_error:.10e}")

## Three-digit chopping and rounding

The course uses small-digit arithmetic to make finite-precision effects visible.
These helper functions simulate decimal chopping and rounding to a specified number of
significant digits.

In [ ]:
def chop(value, digits):
    """Chop a nonzero decimal value to the requested significant digits."""
    if value == 0:
        return 0.0

    exponent = math.floor(math.log10(abs(value)))
    scale = 10 ** (digits - 1 - exponent)
    return math.trunc(value * scale) / scale


def round_sig(value, digits):
    """Round a decimal value to the requested significant digits."""
    if value == 0:
        return 0.0

    exponent = math.floor(math.log10(abs(value)))
    decimals = digits - 1 - exponent
    return round(value, decimals)

In [ ]:
a = 58.5
b = -0.123
c = 1.96

left_to_right = chop(chop(a + b, 3) + c, 3)
reorganized = chop(a + chop(b + c, 3), 3)
exact_value = a + b + c

print(f"exact value        = {exact_value}")
print(f"left-to-right chop = {left_to_right}")
print(f"reorganized chop   = {reorganized}")

## IEEE-754 binary64

Python's ordinary `float` is normally an IEEE-754 binary64 number:

- 1 sign bit,
- 11 exponent bits,
- 52 fraction bits.

For a normalized number,

\[
(-1)^s\,2^{c-1023}(1+f).
\]

In [ ]:
def binary64_fields(value):
    """Return IEEE-754 field bit strings for a Python float."""
    packed = struct.pack(">d", value)
    integer = struct.unpack(">Q", packed)[0]
    bits = f"{integer:064b}"

    return bits[0], bits[1:12], bits[12:]


value = -27.0
sign, exponent, fraction = binary64_fields(value)

print(f"value    = {value}")
print(f"sign     = {sign}")
print(f"exponent = {exponent}")
print(f"fraction = {fraction}")

## Catastrophic cancellation

Subtracting nearly equal floating-point numbers can destroy significant digits.
An algebraically equivalent expression can often be much more stable.

For small \(c>0\),

\[
\sqrt{x^2+c}-x
\]

subtracts two nearby numbers. Rationalizing gives

\[
\frac{c}{\sqrt{x^2+c}+x}.
\]

In [ ]:
x_value = 1.0e8
c_value = 1.0

unstable = math.sqrt(x_value**2 + c_value) - x_value
stable = c_value / (math.sqrt(x_value**2 + c_value) + x_value)

print(f"direct formula      = {unstable:.17g}")
print(f"rationalized formula= {stable:.17g}")

### References

- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.

[← Taylor Series](01_taylor_series.ipynb) · [Next: Bisection →](03_bisection_method.ipynb)